In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('PS_2026.04.07_04.58.49.csv', comment='#', low_memory=False)

df = df.sort_values(by=['pl_name', 'default_flag'], ascending=[True, False])

df_final = df.groupby('pl_name').first().reset_index()

cols_to_keep = [
    'pl_name', 'hostname', 'discoverymethod', 'disc_year', 'disc_facility',
    'pl_rade', 'pl_bmasse', 'st_teff', 'sy_dist', 'default_flag'
]
df_final = df_final[cols_to_keep]

df_final.rename(columns={
    'pl_name': 'planet_name',
    'hostname': 'star_name',
    'discoverymethod': 'discovery_method',
    'disc_year': 'discovery_year',
    'disc_facility': 'facility',
    'pl_rade': 'radius_earth_units',
    'pl_bmasse': 'mass_earth_units',
    'st_teff': 'star_temp_k',
    'sy_dist': 'distance_pc'
}, inplace=True)

df_final['data_reliability'] = df_final['default_flag'].map({1: 'Confirmed Study', 0: 'Preliminary Study'})

numeric_cols = ['radius_earth_units', 'mass_earth_units', 'star_temp_k', 'distance_pc']
for col in numeric_cols:
    df_final[col] = df_final.groupby('discovery_method')[col].transform(lambda x: x.fillna(x.median()))
    df_final[col] = df_final[col].fillna(df_final[col].median())

def get_planet_category(r):
    if r < 0.8: return 'Sub-Earth'
    if 0.8 <= r <= 1.25: return 'Earth-size'
    if 1.25 < r <= 2.0: return 'Super-Earth'
    return 'Giant'

df_final['planet_type'] = df_final['radius_earth_units'].apply(get_planet_category)

def evaluate_habitability(row):
    temp_check = 4000 <= row['star_temp_k'] <= 6500
    size_check = 0.7 <= row['radius_earth_units'] <= 1.5

    if temp_check and size_check:
        return 'High'
    elif temp_check or size_check:
        return 'Medium'
    return 'Low'

df_final['habitability_potential'] = df_final.apply(evaluate_habitability, axis=1)

df_final.drop(columns=['default_flag'], inplace=True)

df_final.to_csv('exoplanet_analysis_final.csv', index=False)

print("Process Completed.")
print(f"Total Unique Planets: {len(df_final)}")
print(f"High Potential Planets Found: {len(df_final[df_final['habitability_potential'] == 'High'])}")

Process Completed.
Total Unique Planets: 6153
High Potential Planets Found: 750


In [2]:
import pandas as pd

df = pd.read_csv('exoplanet_analysis_final.csv')

top_candidates = df[df['habitability_potential'] == 'High'].sort_values(by='distance_pc').head(10)

facility_performance = df[df['habitability_potential'] == 'High']['facility'].value_counts()

yearly_discovery = df[df['habitability_potential'] == 'High'].groupby('discovery_year').size()


planet_type_dist = df['planet_type'].value_counts()

print("--- INSIGHTS REPORT: THE HABITABILITY INDEX ---")
print(f"\n1. Planet Type Distribution:\n{planet_type_dist}")
print(f"\n2. Top 3 Facilities Finding Life Candidates:\n{facility_performance.head(3)}")
print(f"\n3. Closest Potential Earth-Twin: {top_candidates.iloc[0]['planet_name']}")
print(f"   Distance: {top_candidates.iloc[0]['distance_pc']:.2f} Parsecs")

top_candidates.to_csv('top_10_earth_twins.csv', index=False)

--- INSIGHTS REPORT: THE HABITABILITY INDEX ---

1. Planet Type Distribution:
planet_type
Giant          4474
Super-Earth    1123
Earth-size      477
Sub-Earth        79
Name: count, dtype: int64

2. Top 3 Facilities Finding Life Candidates:
facility
Kepler                                          636
K2                                               85
Transiting Exoplanet Survey Satellite (TESS)     26
Name: count, dtype: int64

3. Closest Potential Earth-Twin: HD 219134 f
   Distance: 6.53 Parsecs
